# 1 Importing Packages

How the *MESSAGE_CA* class works: the class will contain all the functions necessary to run the MESSAGEix Canada model. All that you need to run a model is to:

1. Create your scenario folder under `model/scenarios` with its own `config.toml` file

2. Declare the class reading that scenario, for example: `platform, msg, msg_folder = MESSAGE_CA(scenario_name='baseline')`

In [6]:
import os
import sys
import pandas as pd

from model.Model import model_MSG
#from model.Plotter import plotter_MSG
#from model.coders_linkage import CODERS_pull
from model.policies.OBPS import apply_obps
from message_ix import make_df

# Autoreload modules when changes are applied to them
%load_ext autoreload 
%autoreload all

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Sys and OS Operations
os.chdir("../../")

# Scenario Creation and Set-up

Setting up the MESSAGEix scenario using the scenario configuration file and an input file for technology options in technology.yaml

In [5]:
# Set-up
msg = model_MSG(scenario_name='dacs', config=False)
platform, msg_scenario, msg_folder = msg.setup_scenario()



Model Logs: Setting up scenario

Model Logs: Adding technology


		Model creation time: 2 Min and 18 Sec 		
Model Logs: Solving the model...

--- Warning: The GAMS version [48.6.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 03/18/25 10:53:01 48.6.0 f19e462d LEX-LEG x86 64bit/Linux
--- Applying:
    /opt/gams/gams48.6_linux_x64_64_sfx/gmsprmun.txt
--- GAMS Parameters defined
    Input /home/cristianocf/miniconda3/envs/messageix/lib/python3.12/site-packages/message_ix/model/MESSAGE_run.gms
    ScrDir /home/cristianocf/miniconda3/envs/messageix/lib/python3.12/site-packages/message_ix/model/225b/
    SysDir /opt/gams/gams48.6_linux_x64_64_sfx/
    LogOption 4
    --in /home/cristianocf/miniconda3/envs/messageix/lib/python3.12/site-packages/message_ix/model/data/MsgData_MESSAGEix-CA__dacs__v0.gdx
    --out /home/cristianocf/miniconda3/envs/messageix/lib/python3.12/site-packages/message_ix/model/output/MsgOutput_MESSAGEix-CA__dacs__v0.gdx
    --iter /home/cristiano

2025-03-18 10:54:37,594 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2025-03-18 10:54:37,596 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!



				Solving process time: 2 Min and 3 Sec				


## Relations

After a model is solved, if there exist any relations for newly added technologies (added via the scenario configuration), then these need to be added in the next step

In [14]:
# Add relations

DAC_mpen= {
"relation_name" : "DAC_mpen_c",
"growth_act_up" : 0.05,
"initial_act" : 0.12033,
"relation_lower": 0,

# Relation activities by technology
"technologies" : ["DAC_mpen", "dac_ht", "dac_lt"], 
"DAC_mpen_value" : -1,
"dac_ht_value" : 1,
"dac_lt_value" : 1,
}

provinces = provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
            'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
            'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']
years = list(range(2020, 2061, 5))

yearly_relations = []

for year in years:

    dac_ht_df = make_df(
        "relation_activity",
        node_rel=provinces,
        node_loc=provinces,
        technology=['dac_ht'] * len(provinces),
        mode=['M1'] * len(provinces),
        value=[1] * len(provinces),
        relation=[DAC_mpen["relation_name"]] * len(provinces),
        unit=["-"] * len(provinces),
        year_rel=[year] * len(provinces),
        year_act=[year] * len(provinces),
    )
    yearly_relations.append(dac_ht_df)

    dac_lt_df = make_df(
        "relation_activity",
        node_rel=provinces,
        node_loc=provinces,
        technology=['dac_lt'] * len(provinces),
        mode=['M1'] * len(provinces),
        value=[1] * len(provinces),
        relation=[DAC_mpen["relation_name"]] * len(provinces),
        unit=["-"] * len(provinces),
        year_rel=[year] * len(provinces),
        year_act=[year] * len(provinces),
    )
    yearly_relations.append(dac_lt_df)
    
    dac_mpen_df = make_df(
        "relation_activity",
        node_rel=provinces,
        node_loc=provinces,
        technology=['DAC_mpen'] * len(provinces),
        mode=['M1'] * len(provinces),
        value=[-1] * len(provinces),
        relation=[DAC_mpen["relation_name"]] * len(provinces),
        unit=["-"] * len(provinces),
        year_rel=[year] * len(provinces),
        year_act=[year] * len(provinces),
    )
    yearly_relations.append(dac_mpen_df)

# Concatenate all DataFrames
combined_relation_activities = pd.concat(yearly_relations, ignore_index=True)



In [15]:
# Data viewer
combined_relation_activities
#yearly_relations[0]

,relation,node_rel,year_rel,node_loc,technology,year_act,mode,value,unit
0,DAC_mpen_c,Ontario,2020,Ontario,dac_ht,2020,M1,1,-
1,DAC_mpen_c,Quebec,2020,Quebec,dac_ht,2020,M1,1,-
2,DAC_mpen_c,BritishColumbia,2020,BritishColumbia,dac_ht,2020,M1,1,-
3,DAC_mpen_c,Alberta,2020,Alberta,dac_ht,2020,M1,1,-
4,DAC_mpen_c,Manitoba,2020,Manitoba,dac_ht,2020,M1,1,-
...,...,...,...,...,...,...,...,...,...
346,DAC_mpen_c,NewfoundlandandLabrador,2060,NewfoundlandandLabrador,DAC_mpen,2060,M1,-1,-
347,DAC_mpen_c,PrinceEdwardIsland,2060,PrinceEdwardIsland,DAC_mpen,2060,M1,-1,-
348,DAC_mpen_c,NorthwestTerritories,2060,NorthwestTerritories,DAC_mpen,2060,M1,-1,-
349,DAC_mpen_c,Yukon,2060,Yukon,DAC_mpen,2060,M1,-1,-


In [18]:
msg_scenario.remove_solution()
msg_scenario.check_out()

# Add the combined DataFrame to the scenario
msg_scenario.add_par('relation_activity', combined_relation_activities)

# Loop through provinces and years to add the activity_up parameters
for province in provinces:
    for year in years:
        for technology in DAC_mpen["technologies"]:
            
            msg_scenario.add_par('growth_activity_up', 
                        [province, technology, year, "year"], 
                        DAC_mpen["growth_act_up"], 'GWa')  # Assuming 'GWa' is the unit, adjust if needed
            
            msg_scenario.add_par('initial_activity_up', 
                    [province, technology, year, "year"], 
                    DAC_mpen["initial_act"], 'GWa')
        

In [21]:
# Solve model with added relations
#msg_scenario.check_out()
msg_scenario.solve(solve_options={"write_model_to_file":"true", "writelp":"true", "write_model_file":"true", "lpmethod":2}, case="DACS_with_mpen")

--- Warning: The GAMS version [48.6.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 03/18/25 12:33:11 48.6.0 f19e462d LEX-LEG x86 64bit/Linux
--- Applying:
    /opt/gams/gams48.6_linux_x64_64_sfx/gmsprmun.txt
--- GAMS Parameters defined
    Input /home/cristianocf/miniconda3/envs/messageix/lib/python3.12/site-packages/message_ix/model/MESSAGE_run.gms
    ScrDir /home/cristianocf/miniconda3/envs/messageix/lib/python3.12/site-packages/message_ix/model/225b/
    SysDir /opt/gams/gams48.6_linux_x64_64_sfx/
    LogOption 4
    --in /home/cristianocf/miniconda3/envs/messageix/lib/python3.12/site-packages/message_ix/model/data/MsgData_DACS_with_mpen.gdx
    --out /home/cristianocf/miniconda3/envs/messageix/lib/python3.12/site-packages/message_ix/model/output/MsgOutput_DACS_with_mpen.gdx
    --iter /home/cristianocf/miniconda3/envs/messageix/lib/python3.12/site-packages/message_ix/model/output/MsgIterationReport_DACS_with_mpen.gdx
Licensee: MM CIVE 595                  

2025-03-18 12:34:51,249 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2025-03-18 12:34:51,249 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!
